In [0]:
import logging
from pyspark.sql import functions as F

logging.basicConfig(level=logging.INFO)

try:
    logging.info("Silver started")

    df = spark.table("fraud_catalog.bronze.transactions_raw")

    logging.info(f"Rows read: {df.count()}")

except Exception as e:
    logging.error(str(e))
    raise

INFO:root:Silver started
INFO:root:Rows read: 1048575


In [0]:
# removing unwanted columns
for c in df.columns:
    if "unnamed" in c.lower():
        df = df.drop(c)

logging.info("Dropped unwanted columns")

INFO:root:Dropped unwanted columns


In [0]:
# renaming columns
df = df.withColumnRenamed("first", "first_name") \
       .withColumnRenamed("last", "last_name") \
       .withColumnRenamed("long", "cust_long") \
       .withColumnRenamed("lat", "cust_lat")

logging.info("Renamed columns")

INFO:root:Renamed columns


In [0]:
# converting data types and create timestamp

df = df.withColumn("amt", F.col("amt").cast("double")) \
       .withColumn("unix_time", F.col("unix_time").cast("long")) \
       .withColumn("txn_ts", F.from_unixtime("unix_time").cast("timestamp"))

logging.info("Data types converted and timestamp created")

INFO:root:Data types converted and timestamp created


In [0]:
# creating derived columns

df = df.withColumn("txn_date", F.to_date("txn_ts")) \
       .withColumn("txn_hour", F.hour("txn_ts")) \
       .withColumn("txn_day", F.date_format("txn_ts", "E"))

# creating simple flags

df = df.withColumn("is_high_value", F.when(F.col("amt") > 1000, 1).otherwise(0)) \
       .withColumn("is_night_txn", F.when((F.col("txn_hour") >= 22) | (F.col("txn_hour") <= 5), 1).otherwise(0))

logging.info("Derived columns and flags created")

INFO:py4j.clientserver:Received command c on object id p0
INFO:root:Derived columns and flags created


In [0]:
# removing duplicate transactions
before_count = df.count()

df = df.dropDuplicates(["trans_num"])

after_count = df.count()

logging.info(f"Duplicates removed: {before_count - after_count}")

# splitting valid and invalid records
valid_df = df.filter((F.col("amt") > 0) &(F.col("cc_num").isNotNull()) &(F.col("txn_ts").isNotNull()))

invalid_df = df.filter((F.col("amt") <= 0) |(F.col("cc_num").isNull()) |(F.col("txn_ts").isNull()))

logging.info(f"Valid rows: {valid_df.count()}")
logging.info(f"Invalid rows: {invalid_df.count()}")

INFO:root:Duplicates removed: 0
INFO:root:Valid rows: 1048575
INFO:root:Invalid rows: 0


In [0]:
# adding reason for invalid records
invalid_df = invalid_df.withColumn("dq_issue",F.when(F.col("amt") <= 0, "invalid_amount")
     .when(F.col("cc_num").isNull(), "missing_customer")
     .when(F.col("txn_ts").isNull(), "bad_timestamp")
     .otherwise("unknown"))

logging.info("DQ issue column added")

# saving valid data to Silver table
valid_df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("path", "abfss://fraudexternal@fraudstorageacc1234.dfs.core.windows.net/silver/transactions_clean") \
  .saveAsTable("fraud_catalog.silver.transactions_clean")

logging.info("Silver table saved")

# saving invalid data to DQ table
invalid_df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("path", "abfss://fraudexternal@fraudstorageacc1234.dfs.core.windows.net/dq/transactions_invalid") \
  .saveAsTable("fraud_catalog.dq.transactions_invalid")

logging.info("DQ table saved")
logging.info("Silver completed")

INFO:root:DQ issue column added
INFO:root:Silver table saved
INFO:root:DQ table saved
INFO:root:Silver completed
